# 14.03 · When (and when not) to fine-tune

**Module 14 · Local Models & Fine-Tuning** · Part 2 (Pro) · elective

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Lessons 14.01–14.02, Module 10 (confidence intervals, paired comparisons, Lesson 10.04 classifiers). |
| 🔑 Keys | none required · optional `GROQ_API_KEY` (≈ 10K tokens: 100 tickets in 5 batched calls) · optional Ollama |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Choose** the cheapest lever that solves a problem: prompt → few-shot → RAG → tools → fine-tune, and say why.
2. **Build** an honest benchmark: split by template family, check for leaks, keep a test set you look at once.
3. **Measure** classic baselines (naive Bayes, logistic regression in numpy) and few-shot LLMs on the same 234 tickets, with confidence intervals and slices.
4. **Compare** two routers with a paired test instead of eyeballing two accuracies.
5. **Check** calibration and turn error costs into an escalation threshold for a local-first router.
6. **Prepare** training data in the chat format a QLoRA run needs (Lesson 14.04), and name what fine-tuning can't fix.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [OpenAI: model optimization](https://platform.openai.com/docs/guides/model-optimization) | Prompting, evals and fine-tuning as one loop; when each one helps |
| [PEFT](https://huggingface.co/docs/peft/index) | What LoRA adapters change (and don't) in a model |
| [Unsloth: fine-tuning guide](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide) | When fine-tuning is worth it, dataset size and format |
| [TRL dataset formats](https://huggingface.co/docs/trl/dataset_formats) | The `{"messages": [...]}` conversational format used in Part 6 |
| [Smithtune (LangSmith)](https://docs.langchain.com/langsmith/smithtune) | Fine-tuning from traces, compared in an experiment (beta, paid providers) |

> **Why this matters at work.** Orbita Cloud's support lead wants tickets routed to six queues automatically. Someone proposes "let's fine-tune a model". Before renting a GPU, the team needs three numbers: how good is a 10-line classic baseline, how good is a prompted LLM, and what does a misrouted ticket cost compared with an extra model call. This lesson produces those numbers on 900 tickets; 14.04 then fine-tunes and has to beat them.

> 🖥️ **CPU-only is fine.** The baselines train in under a second in numpy. The local LLM results are **recorded** (Qwen3 4B and 0.6B through Ollama on an 8-core CPU, 2026-10-16), so the comparison runs offline; optional cells re-run them on your machine (about 5–8 minutes for 234 tickets on a CPU).

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langchain-ollama>=1.1.0,<2" "numpy>=2.3.0,<3"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain-core", "langchain-groq", "numpy",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/14-local-models-and-fine-tuning/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
LANGSMITH_PROJECT = 'm14-when-to-fine-tune'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config, look for a local Ollama.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.local import ollama_status

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_HF_TOKEN = is_set("HF_TOKEN")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m14-<lesson>: one tracing project per lesson
OLLAMA = ollama_status(timeout=3.0)                              # never raises; 3 s if nothing is listening
HAVE_OLLAMA = OLLAMA.running

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast, "(Groq)")
print("local     :", settings.model_local, "| embeddings:", settings.model_local_embed)
print("ollama    :", f"{OLLAMA.version} at {OLLAMA.base_url}, models: {OLLAMA.names() or 'none pulled'}" if HAVE_OLLAMA
      else f"not running at {OLLAMA.base_url}")
if not HAVE_OLLAMA:
    print("\n⚠️ No Ollama server: 🖥️ local cells will print ⏭️ Skipped. Start the Ollama app (or `ollama serve`) and re-run.")
if not HAVE_GROQ_KEY:
    print("⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

import time

from mastery.evals.stats import find_leaks, paired_compare, wilson_interval
from mastery.finetune import (QUEUES, RECORDED_DIR, LogReg, NaiveBayes, accuracy, best_threshold, by_split, calibration,
                              correct_map, expected_calibration_error, family_overlap, fewshot_prompt, load_predictions,
                              load_tickets, macro_f1, pick_examples, route_curve, top_confusions, write_chat_jsonl)

TICKETS = load_tickets()
SPLIT = by_split(TICKETS)
TRAIN, DEV, TEST = SPLIT["train"], SPLIT["dev"], SPLIT["test"]
GOLD = {t["id"]: t["queue"] for t in TEST}
OUT = REPO_ROOT / "data" / "outputs" / "m14"          # git-ignored
print({k: len(v) for k, v in SPLIT.items()})

---
## Part 1 · The decision tree: the cheapest lever first

| Problem | Try first | Fine-tune only if… |
|---|---|---|
| The model doesn't follow the **format** | Structured output (Module 01), a clearer prompt | You need a very specific style at high volume and prompting is too long/expensive |
| It doesn't know **your facts** (prices, docs, today's incident) | **RAG** (Module 06) or a **tool** (Module 02) | Never for facts that change: a fine-tune freezes them on training day |
| It must **act** (look up an invoice, reset a password) | Tools and an agent | — |
| A **narrow, stable classification** at high volume (routing, screening, tagging) | Few-shot prompt, then a classic baseline | The small model must run locally/cheaply and few-shot isn't accurate enough: **this is the case for fine-tuning** |
| It must be **smaller or faster** (on-device, privacy, cost) | A smaller model + few-shot | You can show the small tuned model matches the big one on *your* test set |

Ticket routing is the textbook fine-tuning case: six stable labels, thousands of tickets a day, privacy pressure to keep them on the machine. That's exactly why we measure the alternatives first.

---
## Part 2 · An honest benchmark: families, slices and leaks

In [ ]:
# 🧪 900 synthetic tickets: six queues, English and Spanish, a "hard" slice (very short, two intents, sarcasm).
rows = []
for name, part in SPLIT.items():
    rows.append({"split": name, "tickets": len(part), "families": len({t["family"] for t in part}),
                 **{q: sum(t["queue"] == q for t in part) for q in QUEUES},
                 "spanish": sum(t["lang"] == "es" for t in part), "hard": sum(t["hard"] for t in part)})
table(rows, ["split", "tickets", "families", *QUEUES, "spanish", "hard"])
print()
for t in [t for t in TEST if t["hard"]][:3] + [t for t in TEST if t["lang"] == "es"][:2]:
    print(f"{t['queue']:<16} {t['text']}")

**You should now see** train 594 / dev 72 / test 234, balanced queues (99 / 12 / 39 each), 54 Spanish and 36 hard test tickets, then five examples such as `outage  down again?? ...` and a Spanish billing ticket.

### What just happened?

Every ticket comes from a **template family** (one sentence pattern with different products, amounts, greetings and typos). A whole family goes to one split, chosen by a salted hash within each (queue, language, hard) stratum. If families were split randomly, the test set would hold near-copies of training tickets and every model would look better than it will on new tickets.

In [ ]:
# 🧪 Two leak checks before any number is trusted.
print("families in both train and test:", family_overlap(TRAIN, TEST) or "none")
EXAMPLES = pick_examples(TRAIN, per_queue=2)                     # the few-shot examples of Part 4
leaks = find_leaks(fewshot_prompt(EXAMPLES), {t["id"]: t["text"] for t in TEST}, n=6)
print("test tickets sharing ≥ 6 words with the few-shot prompt:", len(leaks))
for leak in leaks[:3]:
    print("  ", leak)

**You should now see** `none` for shared families and a small number of shared 6-word passages (or 0), each a generic phrase rather than a copied ticket.

### What just happened?

`family_overlap` checks the split; `find_leaks` (Module 10) checks the **prompt**: few-shot examples copied from the test set are the most common way a prompted model looks better than it is. Always run both before reporting a score.

---
## Part 3 · Classic baselines in numpy

In [ ]:
# 🧪 Multinomial naive Bayes and logistic regression on hashed word + character n-grams. No scikit-learn.
texts, labels = [t["text"] for t in TRAIN], [t["queue"] for t in TRAIN]
BASELINES, CONF = {}, {}
for name, model in {"naive Bayes": NaiveBayes(), "logistic regression": LogReg()}.items():
    start = time.perf_counter()
    model.fit(texts, labels)
    fit_s = time.perf_counter() - start
    start = time.perf_counter()
    preds = model.predict([t["text"] for t in TEST])
    per_ticket_ms = (time.perf_counter() - start) / len(TEST) * 1000
    BASELINES[name] = {t["id"]: p for t, (p, _) in zip(TEST, preds)}
    CONF[name] = {t["id"]: c for t, (_, c) in zip(TEST, preds)}
    print(f"{name:<20} trained in {fit_s:.2f} s, {per_ticket_ms:.2f} ms per ticket")


def report(readers):
    rows = []
    for name, pred in readers.items():
        def sl(keep):
            ids = [t["id"] for t in TEST if keep(t)]
            return f"{sum(pred.get(i) == GOLD[i] for i in ids) / len(ids):.0%}"
        rows.append({"router": name, "accuracy [95% CI]": f"{accuracy(pred, GOLD):.1%}".split(" (")[0],
                     "macro-F1": f"{macro_f1(pred, GOLD, QUEUES):.2f}", "English": sl(lambda t: t["lang"] == "en"),
                     "Spanish": sl(lambda t: t["lang"] == "es"), "hard": sl(lambda t: t["hard"])})
    table(rows, ["router", "accuracy [95% CI]", "macro-F1", "English", "Spanish", "hard"])


report(BASELINES)
print("\nlogistic regression's top mistakes (gold → predicted):", top_confusions(BASELINES["logistic regression"], GOLD, 3))

**You should now see** both baselines trained in well under a second, predicting in a fraction of a millisecond per ticket, at about **60%** accuracy (95% CI roughly 54–69%), around 40% on Spanish, and mistakes such as `account_access → bug`.

### What just happened?

60% on six balanced queues is far above chance (17%) and far below useful. The baselines learn **words**: the test families use sentence patterns they never saw, and only one Spanish family per queue was in training. That's the honest number a fine-tune must beat; with random splits it would read above 90% and mislead everyone.

---
## Part 4 · Few-shot LLMs on the same tickets

> 🖥️ Runs **on your computer** through Ollama: no key, no cost, nothing leaves the machine. On a **CPU-only** laptop (no NVIDIA GPU) expect 10–60 s for the first call (the model loads into RAM) and roughly 5–15 tokens per second for a 4B model afterwards; the course gives local calls a 300 s timeout. Without a running Ollama it prints `⏭️ Skipped`.

In [ ]:
# 🧪 Recorded few-shot runs (same prompt, 12 examples, batches of 20) of two local models on all 234 tickets.
RECORDED = {}
for file in ["fewshot-qwen3-4b.json", "fewshot-qwen3-0.6b.json"]:
    rec = load_predictions(RECORDED_DIR / file)
    RECORDED[rec["model"]] = rec["labels"]
    print(f"{rec['model']:<20} recorded {rec['recorded']} on {rec['hardware']}: {rec['seconds'] / 60:.1f} min, "
          f"{rec['input_tokens'] + rec['output_tokens']:,} tokens, {rec['failed_batches']} failed batches")
print()
report({**BASELINES, **RECORDED})

**You should now see** the recorded `ollama:qwen3:4b` far ahead of the baselines (about 94%, Spanish and hard tickets included), `ollama:qwen3:0.6b` **below** them (about 47%), and how long the CPU took (6.4 and 1.6 minutes for 234 tickets).

### What just happened?

`route_with_llm` sends 20 tickets per request with a strict schema (`{"labels": [{"id", "queue"}]}`), so the instructions and examples are paid once per batch. The 4B model generalizes to sentence patterns and Spanish it never saw in the examples; that's the gap between a language model and a bag of words. The 0.6B model is four times faster but worse than a numpy baseline: smaller isn't automatically "good enough", which is the gap a fine-tune of a small model (14.04) tries to close.

In [ ]:
# 💸 (HAVE_GROQ_KEY) The hosted reference: gpt-oss-20b on the first 100 test tickets, 5 batched calls (≈ 10K tokens).
from mastery.finetune import route_with_llm, save_predictions
from mastery.models import get_model

LIVE_N = 100   # ← keep it small: the free tier allows 200K tokens a day per model
if HAVE_GROQ_KEY:
    groq_run = route_with_llm(get_model("fast", temperature=0), TEST[:LIVE_N], examples=EXAMPLES,
                              on_batch=lambda i, r: print(f"batch {i}: {r.tokens:,} tokens so far"))
    RECORDED[f"groq:{settings.model_fast}"] = groq_run.labels
    save_predictions(OUT / "fewshot-groq.json", f"groq:{settings.model_fast}", groq_run.labels, split=f"test[:{LIVE_N}]",
                     input_tokens=groq_run.input_tokens, output_tokens=groq_run.output_tokens, seconds=groq_run.seconds)
    print(f"{groq_run.seconds:.1f} s, {groq_run.tokens:,} tokens, {groq_run.failed_batches} failed batches → {OUT / 'fewshot-groq.json'}")
    sub = {i: GOLD[i] for i in groq_run.labels}
    for name in [f"groq:{settings.model_fast}", "ollama:qwen3:4b", "logistic regression"]:
        pred = RECORDED.get(name) or BASELINES[name]
        print(f"  {name:<28} {accuracy(pred, sub):.1%}".split(" (")[0], "on the same", len(sub))
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

> 💸 With `GROQ_API_KEY`: 5 calls, about 10K tokens (gpt-oss also spends output tokens on hidden reasoning). Saved to `data/outputs/m14/fewshot-groq.json` for Project 14.

**You should now see** five `batch` lines and the three routers on the same 100 tickets: gpt-oss-20b and qwen3:4b both around 90%+, logistic regression around 60%.

In [ ]:
# 🖥️ (HAVE_OLLAMA, RUN_STRETCH) Re-run the local few-shot router on YOUR machine (40 tickets ≈ 1–2 min on a CPU).
RUN_STRETCH = False
LOCAL = settings.model_local
if HAVE_OLLAMA and RUN_STRETCH and OLLAMA.get(LOCAL):
    mine = route_with_llm(get_model("fast", "ollama", temperature=0), TEST[:40], examples=EXAMPLES)
    sub = {i: GOLD[i] for i in mine.labels}
    print(f"{LOCAL}: {accuracy(mine.labels, sub):.0%} in {mine.seconds:.0f} s on this machine "
          f"(recorded: {accuracy(RECORDED['ollama:qwen3:4b'], sub):.0%})")
elif HAVE_OLLAMA and RUN_STRETCH:
    print(f"⏭️ Skipped: {LOCAL} is not pulled (ollama pull {LOCAL}).")
else:
    print("⏭️ Skipped (optional): needs Ollama and RUN_STRETCH = True.")

**You should now see** `⏭️ Skipped (optional)` by default; with Ollama and `RUN_STRETCH = True`, your accuracy and time on 40 tickets next to the recorded number.

### Is the difference real? A paired comparison

In [ ]:
# 🧪 Same 234 tickets, scored 1/0 for each router: a paired test that only uses the tickets where they disagree.
lr = correct_map(BASELINES["logistic regression"], GOLD)
q4 = correct_map(RECORDED["ollama:qwen3:4b"], GOLD)
print(paired_compare(lr, q4, names=("logistic regression", "qwen3:4b few-shot")))
print()
q06 = correct_map(RECORDED["ollama:qwen3:0.6b"], GOLD)
print(paired_compare(lr, q06, names=("logistic regression", "qwen3:0.6b few-shot")))

**You should now see** qwen3:4b winning 78 tickets and losing 4 (+32 points, p ≈ 1e-18), and the 0.6B model *losing* to logistic regression (−16 points, p < 0.001): both differences are real.

### What just happened?

Two accuracies with overlapping intervals can still be a clear win: pairing (Module 10) only looks at tickets where the routers **disagree**. This is the comparison 14.04 repeats for the fine-tuned model.

---
## Part 5 · Calibration and a local-first router

In [ ]:
# 🧪 Does "80% confident" mean right 80% of the time? Logistic regression's reliability table.
ids = list(GOLD)
conf = [CONF["logistic regression"][i] for i in ids]
right = [lr[i] for i in ids]
table(calibration(conf, right, bins=5), ["bin", "n", "confidence", "accuracy"])
print("ECE:", expected_calibration_error(conf, right))

**You should now see** most tickets below 0.6 confidence, accuracy **above** the stated confidence in the middle bins (0.48 → 81%, 0.70 → 97%: under-confident), a top bin of five tickets that are all wrong, and an ECE of about 0.24.

### What just happened?

A softmax over six queues with L2 spreads probability, so 0.48 can still be right four times out of five. What matters for routing is that **higher confidence mostly means more often right**, so a threshold separates easy tickets from hard ones. The five confidently wrong tickets are a warning: a few words ("export", "login") can push the baseline to the wrong queue with high confidence, so never let a baseline's confidence skip human review on high-stakes queues. (Lesson 10.04 calibrates judges the same way.)

In [ ]:
# 🧪 Tier 1: logistic regression in-process (free, instant). Below the threshold: escalate to qwen3:4b on the laptop.
COST_WRONG = 0.50      # ← a misrouted ticket: a triage person re-routes it (≈ 1 min), in USD
COST_ESCALATE = 0.10   # ← one ticket through the local LLM: ≈ 2 s of a busy CPU and a slower first reply, in USD
points = route_curve(conf, right, [q4[i] for i in ids], cost_wrong=COST_WRONG, cost_escalate=COST_ESCALATE)
table([p.__dict__ for p in points[::2]], ["threshold", "kept", "escalated", "accuracy", "cost", "cost_per_ticket"])
best = best_threshold(points)
print(f"\nbest threshold {best.threshold}: keep {best.kept}, escalate {best.escalated}, accuracy {best.accuracy:.0%}, "
      f"${best.cost_per_ticket:.3f} per ticket (all local LLM: ${points[-1].cost_per_ticket:.3f})")

**You should now see** a table from threshold 0.0 (keep everything, 63% accuracy) to 1.0 (escalate everything, 94%), and a best threshold of about **0.45**: the baseline keeps 106 tickets, 128 go to the LLM, accuracy 90%, at $0.106 per ticket against $0.128 for "all LLM".

### What just happened?

The threshold isn't a feeling; it's the minimum of `wrong × cost_wrong + escalated × cost_escalate`. Make a misroute expensive (`COST_WRONG = 2.00`) and the answer becomes "escalate everything": the LLM is simply better. Make the LLM expensive (a hosted model with a privacy review) and the threshold drops. A fine-tuned small model (14.04) changes the curve itself: higher local accuracy at lower cost per ticket. The same structure screens tool calls or prompts in front of a bigger model.

---
## Part 6 · Training data for 14.04, and what fine-tuning can't fix

In [ ]:
# 🧪 Export train and dev in the conversational format TRL/Unsloth read. The test set stays out.
train_file = write_chat_jsonl(TRAIN, OUT / "router-train.jsonl")
dev_file = write_chat_jsonl(DEV, OUT / "router-dev.jsonl")
print(train_file.name, sum(1 for _ in train_file.open(encoding="utf-8")), "examples |", dev_file.name,
      sum(1 for _ in dev_file.open(encoding="utf-8")))
print(train_file.open(encoding="utf-8").readline()[:400])

**You should now see** `router-train.jsonl 594 examples | router-dev.jsonl 72` and one line `{"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", "content": "<queue>"}]}`.

### What just happened?

The dev set has no Spanish or hard tickets (those strata have only two families, one for train and one for test), so use it for quick checks and keep the slices for test.

Each example teaches the model one mapping: this ticket → this queue, in a few tokens. 594 examples is small for fine-tuning but enough for a narrow label task with LoRA; 14.04 uploads these files to Colab. The test tickets never go into training, so 14.04's score is comparable with every row in Part 4.

**What fine-tuning can't fix:**

- **Fresh knowledge.** If Orbita launches "Pulse" sensors next month, a tuned model routes their tickets only as well as the words suggest; the *facts* (prices, docs, incidents) belong in RAG or tools, which change without retraining.
- **A bad label scheme.** If humans disagree on `bug` vs `outage`, the model learns the disagreement. Fix the definitions first (Part 4's prompt doubles as the labelling guide).
- **Drift.** New products and phrasings lower accuracy over time: keep the test set, re-score monthly, retrain when it drops (Module 12 monitoring).

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| 95%+ for every model | Random split: near-duplicate tickets in train and test | Split by family/template/customer; run `family_overlap` |
| Few-shot looks great, production doesn't | Examples copied from the test set | `find_leaks(prompt, test_texts)` before reporting |
| `error` labels in a run | A batch failed twice (rate limit, malformed reply) | Counted as wrong on purpose; rerun just those ids, or a smaller `batch_size` |
| `429` on Groq | Free-tier tokens per minute | Smaller `LIVE_N`, wait a minute; the recorded files make the lesson work without it |
| Local run takes forever | CPU, thinking on, or a big model | The course sets `reasoning=False`; use `qwen3:4b`, fewer tickets |
| Threshold "best" is 0 or 1 | Costs make one side always cheaper | That's an answer too: check the costs with the people who pay them |

## ⚠️ Production notes

> ⚠️ **Production note: freeze the test set, version the prompt.** (Parts 2, 4)
> Store test ids, prompt text and model version with every recorded run (the JSON files here do). A score without them can't be reproduced or compared next quarter.

> ⚠️ **Production note: a router needs a human lane.** (Part 5)
> Below the second threshold (both tiers unsure), send the ticket to a triage person, and log disagreements between tiers: they are your next training examples.

## 🏋️ Exercises

1. **(Easy)** Set `COST_WRONG = 0.20` and rerun Part 5. How does the best threshold move, and why?
2. **(Medium)** Report accuracy per queue for qwen3:4b and logistic regression. Which queue does the LLM fix most?
3. **(Medium)** Train `LogReg` on train **and** dev (666 tickets). Does test accuracy change beyond its CI?
4. **(Stretch)** Build a two-tier router where tier 2 is gpt-oss-20b on Groq (use your `fewshot-groq.json` ids only) and compare its cost curve with the local one using a privacy cost of $0.05 per escalated ticket.

<details>
<summary><b>Show solutions</b></summary>

**1.** A cheaper mistake makes escalation relatively expensive, so the threshold falls to 0.35: 165 tickets stay with the baseline, accuracy 81%, lower cost. With `COST_WRONG = 2.00` it rises to 0.9: escalate everything.

**2.** `for q in QUEUES: ids = [i for i, g in GOLD.items() if g == q]; print(q, sum(RECORDED["ollama:qwen3:4b"][i] == q for i in ids) / len(ids), sum(BASELINES["logistic regression"][i] == q for i in ids) / len(ids))`. Expect the largest gains on `how_to` vs `feature_request` and on Spanish tickets.

**3.** `LogReg().fit([t["text"] for t in TRAIN + DEV], [t["queue"] for t in TRAIN + DEV])`; usually a few points, inside the interval: more examples of the same families help less than new families.

**4.** `g = load_predictions(OUT / "fewshot-groq.json")["labels"]; sub = list(g)`, then `route_curve([CONF["logistic regression"][i] for i in sub], [lr[i] for i in sub], [int(g[i] == GOLD[i]) for i in sub], cost_wrong=2.0, cost_escalate=0.05 + 0.0001)`.
</details>

## 🔎 Docs drill (5 min)

In TRL's dataset formats, what is the difference between the *conversational* and the *standard* format, and which one does Part 6 write?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [TRL dataset formats](https://huggingface.co/docs/trl/dataset_formats).
2. **Standard**: plain text fields (`"text"`, or `"prompt"`/`"completion"` strings). **Conversational**: lists of `{"role", "content"}` messages, rendered with the model's chat template.
3. Part 6 writes the **conversational language-modeling** format: one `"messages"` list per example.

</details>

## ✅ Checkpoint

- [ ] I can say which lever to try first for format, facts, actions and narrow classification.
- [ ] My split is by family, and I ran both leak checks.
- [ ] I know the baseline, the few-shot local and the hosted numbers on the same test set, with CIs.
- [ ] I can read a paired comparison and a calibration table, and set a threshold from costs.
- [ ] `router-train.jsonl` and `router-dev.jsonl` exist in `data/outputs/m14/` for Lesson 14.04.

## 🔎 Research checkpoint

Read the [Unsloth fine-tuning guide](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide) and log in [`docs/research-log.md`](../../docs/research-log.md): its advice on dataset size for classification, and one case where it recommends **not** fine-tuning.

## Summary

- Fine-tune last: prompts, few-shot, RAG and tools are cheaper and change without retraining.
- Honest benchmarks split by family and check prompts for leaks; otherwise every model looks great.
- On these tickets a numpy baseline gets ~60%, a few-shot 4B local model ~90%, at a CPU cost of seconds per batch.
- Pair your comparisons and pick thresholds from error costs, not intuition.
- Fine-tuning teaches behaviour on stable labels; facts, label confusion and drift need other fixes.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 14-04-qlora-on-colab.ipynb** → [`14-04-qlora-on-colab.ipynb`](14-04-qlora-on-colab.ipynb).

[Module 14 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)